# 4주차 | 텍스트 분류 — 긍정/부정 자동 판단

```
리뷰 텍스트 → 전처리 → TF-IDF → 머신러닝 모델 → 긍정 😊 / 부정 😞
```
---

In [1]:
# 필요한 도구 설치 및 불러오기
!pip install scikit-learn --quiet

import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB   # 나이브 베이즈
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("✅ 준비 완료!")

✅ 준비 완료!


In [4]:
# 학습 데이터: 긍정 25개 + 부정 25개 = 총 50개
# 주의할 점! 1번째 긍정리뷰가 진짜 긍정인지 부정인지
# 정말로 이 문장이 한가지 의미를 내포하고 있는지

긍정_리뷰 = [
    "치킨이 정말 바삭하고 맛있어요 소스도 최고예요",
    "배달이 엄청 빠르고 포장도 꼼꼼했어요 재주문 할게요",
    "피자 도우가 쫄깃하고 치즈가 넘쳐요 완전 맛집",
    "서비스가 친절하고 음식도 맛있어요 강력 추천",
    "가성비 최고예요 양도 많고 맛도 좋아요",
    "국물이 진하고 깊은 맛이에요 또 시킬게요",
    "고기가 부드럽고 신선해요 냄새도 안 나요",
    "빠른 배달에 음식 맛도 훌륭해요 감사합니다",
    "포장이 깔끔하고 음식이 따뜻하게 왔어요",
    "사장님이 너무 친절하세요 음식도 맛있고요",
    "면이 쫄깃하고 국물이 시원해요 자주 시킬게요",
    "닭고기가 촉촉하고 양념이 잘 배어있어요",
    "해산물이 신선하고 볶음밥도 맛있어요",
    "두부가 부드럽고 찌개 간이 딱 맞아요",
    "냉면 육수가 시원하고 고명이 푸짐해요",
    "된장찌개 맛이 구수하고 밥이 윤기나요",
    "떡볶이가 쫄깃하고 소스가 매콤달콤해요",
    "샐러드가 신선하고 드레싱이 맛있어요",
    "갈비찜이 부드럽고 양이 많아서 만족해요",
    "재료가 신선하고 맛이 깔끔해서 자주 시켜요",
    "치즈가 풍부하고 빵이 바삭해서 맛있었어요",
    "배달 포장이 완벽하고 음식도 따뜻하게 왔어요",
    "양념이 골고루 배어있고 육즙이 살아있어요",
    "순두부찌개 국물이 얼큰하고 두부가 신선해요",
    "파스타 면이 탱탱하고 소스가 진해요",
]

부정_리뷰 = [
    "배달이 너무 늦어서 음식이 다 식었어요",
    "치킨이 눅눅하고 기름기가 너무 많아요",
    "포장이 엉망이라 국물이 다 쏟아졌어요",
    "서비스가 불친절하고 음식도 별로였어요",
    "양이 너무 적고 가격은 비싸요 실망이에요",
    "음식에서 이상한 냄새가 났어요 위생이 걱정돼요",
    "주문한 거랑 다른 음식이 왔어요 환불하고 싶어요",
    "맛이 너무 짜고 자극적이에요 먹기 힘들었어요",
    "배달 시간이 두 시간이나 걸렸어요 너무해요",
    "재료가 신선하지 않은 것 같아요 다시는 안 시켜요",
    "면이 퉁퉁 불어있고 국물이 싱거워요",
    "고기가 질기고 냄새가 심하게 나요",
    "두부가 오래된 것 같고 찌개가 너무 짜요",
    "냉면이 식어있고 육수 맛이 이상해요",
    "된장찌개에서 쉰 냄새가 나요 못 먹겠어요",
    "떡이 딱딱하고 소스가 너무 매워서 못 먹겠어요",
    "샐러드 재료가 시들어있고 드레싱이 상한 것 같아요",
    "파스타가 너무 짜고 면이 퍼져있어요",
    "치즈가 굳어있고 빵이 눅눅해서 실망했어요",
    "음식이 차갑게 왔고 포장도 터져있었어요",
    "양념이 안 배어있고 육즙이 없어서 퍽퍽해요",
    "재료가 변질된 것 같아서 먹다가 버렸어요",
    "배달원이 불친절하고 음식도 엉망이었어요",
    "가격 대비 질이 너무 낮아요 다시는 안 시킬게요",
    "조리가 덜 됐는지 속이 안 익었어요",
]

X = 긍정_리뷰 + 부정_리뷰        # 전체 리뷰
y = [1]*25 + [0]*25          # 정답 (1=긍정, 0=부정)

print(f"전체 데이터: {len(X)}개 (긍정 {sum(y)}개 / 부정 {len(y)-sum(y)}개)")

전체 데이터: 50개 (긍정 25개 / 부정 25개)


In [17]:
# ══════════════════════════════════════════
# 핵심 파이프라인 5단계
# ══════════════════════════════════════════

# Step 1. 전처리
def preproc(text):
    text = re.sub(r'[^가-힣a-zA-Z0-9 ]', ' ', text)   # 한글, 숫자, 영어, 공백이 아닌 것은 모두 제거해라 " "으로
    return re.sub(r'\s+', ' ', text).strip()         # \s -> 공백문자 전체 + -> 1개 이상 .strip() -> 양쪽 공백 제거

X = [preproc(r) for r in X]

# Step 2. 데이터 분할(훈련 75% / 테스트 25%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=2026)

# Step 3. TF-IDF 변환
trans = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4))
X_train_vec = trans.fit_transform(X_train)    # 데이터 누수를 방지하기 위해 train data에만 .fit
X_test_vec  = trans.transform(X_test)


# Step 4. 학습 + 예측 (나이브 베이즈)
model = MultinomialNB()
model.fit(X_train_vec, y_train)
pred = model.predict(X_test_vec) # 이 줄에서 'pred' 변수가 숫자 배열로 초기화됩니다.

# Step 5. 평가
print(f"정확도: {accuracy_score(y_test, pred):.2%}")
print()
print(classification_report(y_test, pred, target_names=["부정", "긍정"]))

정확도: 69.23%

              precision    recall  f1-score   support

          부정       1.00      0.50      0.67         8
          긍정       0.56      1.00      0.71         5

    accuracy                           0.69        13
   macro avg       0.78      0.75      0.69        13
weighted avg       0.83      0.69      0.68        13



In [18]:
print(X_train_vec)
# 37 문서 (=50*0.75) / 1157 단어
# Sparse Matrix(희소 행렬) 내에서 0이 아닌 특정 값이 어떤 위치에 저장되어 있는지
# (0, 98) 0.1366...는 첫 번째 문서(행 인덱스 0)의 98번째 특성(열 인덱스 98)에 해당하는 TF-IDF 값이 0.1366...이라는 의미

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 1907 stored elements and shape (37, 1118)>
  Coords	Values
  (0, 98)	0.13663861711020317
  (0, 555)	0.13663861711020317
  (0, 696)	0.13663861711020317
  (0, 385)	0.14231222765973006
  (0, 99)	0.13663861711020317
  (0, 556)	0.13663861711020317
  (0, 697)	0.13663861711020317
  (0, 100)	0.13663861711020317
  (0, 557)	0.13663861711020317
  (0, 261)	0.1657687838476643
  (0, 883)	0.1657687838476643
  (0, 583)	0.1657687838476643
  (0, 551)	0.14872872866649478
  (0, 262)	0.1657687838476643
  (0, 884)	0.1657687838476643
  (0, 584)	0.1657687838476643
  (0, 263)	0.1657687838476643
  (0, 885)	0.1657687838476643
  (0, 24)	0.13663861711020317
  (0, 418)	0.13663861711020317
  (0, 25)	0.13663861711020317
  (0, 15)	0.13663861711020317
  (0, 402)	0.1657687838476643
  (0, 425)	0.053385154189641786
  (0, 16)	0.1657687838476643
  :	:
  (36, 184)	0.13663195272257977
  (36, 722)	0.13663195272257977
  (36, 51)	0.13663195272257977
  (36, 472)	0.1366

In [20]:
# 각 리뷰의 긍정/부정 확률도 확인
proba = model.predict_proba(X_test_vec)

print("번호    부정확률    긍정확률   예측    실제   정오")
print("-" * 48)
for i, (n_p, p_p) in enumerate(proba):
    pred_label = "긍정 " if pred[i] == 1 else "부정 "
    actual   = "긍정" if y_test[i] == 1 else "부정"
    correct   = "✅" if pred[i] == y_test[i] else "❌" # pred[i]는 여전히 숫자 예측 (0 또는 1)
    print(f"  {i+1:2}번   {n_p:.4f}   {p_p:.4f}   {pred_label}  {actual}  {correct}")

번호    부정확률    긍정확률   예측    실제   정오
------------------------------------------------
   1번   0.3867   0.6133   긍정   긍정  ✅
   2번   0.4964   0.5036   긍정   부정  ❌
   3번   0.3548   0.6452   긍정   긍정  ✅
   4번   0.4045   0.5955   긍정   긍정  ✅
   5번   0.4560   0.5440   긍정   부정  ❌
   6번   0.4437   0.5563   긍정   부정  ❌
   7번   0.5099   0.4901   부정   부정  ✅
   8번   0.3566   0.6434   긍정   긍정  ✅
   9번   0.5271   0.4729   부정   부정  ✅
  10번   0.3776   0.6224   긍정   긍정  ✅
  11번   0.5008   0.4992   부정   부정  ✅
  12번   0.6155   0.3845   부정   부정  ✅
  13번   0.3944   0.6056   긍정   부정  ❌


> **확률 해석**  
> - 긍정 확률 **0.9 이상** → 모델이 강하게 확신  
> - 긍정 확률 **0.5 근처** → 모델이 헷갈리는 경우  
> - 긍정 확률 **0.1 이하** → 부정이라고 강하게 확신

---
## Step 3 — TF-IDF 변환

### char n-gram 방식

```python
변환기 = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4))
```

**n-gram이란?** 연속된 n개의 문자 조합

```
"바삭하고" → char n-gram 분해:
  2글자: "바삭", "삭하", "하고"
  3글자: "바삭하", "삭하고"
  4글자: "바삭하고"
```

**한국어에서 효과적인 이유:**

```
한국어는 같은 의미도 형태가 다양하다:
  맛있다 / 맛있어요 / 맛있었어요 / 맛있고
  
공통 패턴 "맛있"이 항상 등장한다
→ KoNLPy 없이도 형태 변화를 자연스럽게 처리
```

---
## Step 4 - 두 가지 분류기

### 1) 나이브 베이즈 (MultinomialNB)

**핵심 아이디어:** 각 패턴이 긍정/부정에 등장하는 확률을 학습한다.

```
학습 결과 예시:
  패턴       긍정 확률   부정 확률
  "맛있"      80%         10%
  "바삭"      75%          8%
  "식었"       5%         70%
  "별로"       3%         65%

새 리뷰 "바삭 맛있다":
  P(긍정) ∝ 0.75 × 0.80 = 0.60  ← 높음!
  P(부정) ∝ 0.08 × 0.10 = 0.008 ← 낮음
  → 긍정 분류
```

"나이브(순진한)" 가정: 각 단어가 서로 독립적이라고 가정한다.  
현실과 맞지 않지만, 계산이 단순하고 잘 작동한다.

### 2) 로지스틱 회귀 (LogisticRegression)

**핵심 아이디어:** 각 패턴에 가중치를 부여해서 점수를 계산한다.

```
점수 = w("맛있")×2.1 + w("바삭")×1.8 + w("식었")×(-2.3) + ...
점수 → 시그모이드 함수 → 0~1 확률 → 분류
```

**강점:** 어떤 패턴이 판단에 영향을 줬는지 가중치로 확인 가능하다.

> - 실전에서는 두 모델 모두 돌려보고 성능이 좋은 것을 선택한다.  
> - "단순한 모델이 항상 나쁜 것은 아니다"

In [24]:
# ── 나이브 베이즈 결과 ──
nb_acc = accuracy_score(y_test, pred)
print(f"나이브 베이즈 정확도: {nb_acc:.2%} ")

# ── 로지스틱 회귀 ──
LR = LogisticRegression(max_iter=1000, random_state=42)
LR.fit(X_train_vec, y_train)
pred_LR = LR.predict(X_test_vec)

lr_acc = accuracy_score(y_test, pred_LR)
print(f"로지스틱 회귀 정확도: {lr_acc:.2%} ")
print()

# 둘 중 어느 쪽이 더 좋은지 비교
if nb_acc >= lr_acc:
    print("→ 이 데이터에서는 나이브 베이즈가 더 좋거나 동일하다")
else:
    print("→ 이 데이터에서는 로지스틱 회귀가 더 좋다")

나이브 베이즈 정확도: 69.23% 
로지스틱 회귀 정확도: 53.85% 

→ 이 데이터에서는 나이브 베이즈가 더 좋거나 동일하다


In [27]:
# 두 모델 상세 성능 비교
print("=" * 55)
print("나이브 베이즈 상세 성능")
print("=" * 55)
print(classification_report(y_test, pred,
                              target_names=["부정", "긍정"],
                              zero_division=0))

print("=" * 55)
print("로지스틱 회귀 상세 성능")
print("=" * 55)
print(classification_report(y_test, pred_LR,
                              target_names=["부정", "긍정"],
                              zero_division=0))

나이브 베이즈 상세 성능
              precision    recall  f1-score   support

          부정       1.00      0.50      0.67         8
          긍정       0.56      1.00      0.71         5

    accuracy                           0.69        13
   macro avg       0.78      0.75      0.69        13
weighted avg       0.83      0.69      0.68        13

로지스틱 회귀 상세 성능
              precision    recall  f1-score   support

          부정       1.00      0.25      0.40         8
          긍정       0.45      1.00      0.62         5

    accuracy                           0.54        13
   macro avg       0.73      0.62      0.51        13
weighted avg       0.79      0.54      0.49        13



## Step 5 - 평가 지표
### 1) 정확도 (Accuracy)

```
정확도: 84.62%
→ 테스트 리뷰 13개 중 11개를 올바르게 분류
```

>  **현실**  
> - 정확도 70%도 실전에서 충분히 유용하다.  
> - 1만 개 리뷰 중 7,000개를 자동 처리하고, 나머지 3,000개만 사람이 검토.  
> - 정확도 100%를 목표로 하기보다 "어느 정도면 실용적인가"

### 2) classification_report

```
              precision    recall  f1-score   support
        부정       0.86      0.86      0.86         7  ← 실제 부정 7개
        긍정       0.83      0.83      0.83         6  ← 실제 긍정 6개
    accuracy                           0.85        13  ← 전체 정확도
   macro avg       0.85      0.85      0.85        13  ← 단순 평균
weighted avg       0.85      0.85      0.85        13  ← 개수 비례 평균
```

| 열 | 의미 |
|---|---|
| `support` | 테스트에 실제로 있는 개수 |
| `precision` | 긍정이라 예측한 것 중 실제 긍정 비율 |
| `recall` | 실제 긍정 중 긍정으로 찾아낸 비율 |
| `f1-score` | precision과 recall의 균형 점수 |

---
## 가중치 분석 — 모델이 본 근거

로지스틱 회귀의 강점: 어떤 패턴이 판단에 영향을 줬는지 확인할 수 있습니다.

```python
가중치[i] > 0  → 이 패턴이 있으면 긍정 쪽으로 판단
가중치[i] < 0  → 이 패턴이 있으면 부정 쪽으로 판단
가중치[i] ≈ 0  → 판단에 거의 영향 없음
```

- 가중치가 이상하게 나왔다면 (예: "맛없다"가 긍정 패턴에 포함) 데이터 레이블에 문제가 있을 가능성이 높다.  
- 가중치 확인은 모델 디버깅의 첫 번째 단계이다.

In [29]:
# 의미 없는 n-gram 패턴 필터
# char_wb n-gram 특성상 조사·어미·공백만 있는 패턴이 섞인다.
# → 실제 의미 있는 단어 패턴만 골라낸다.

어미_조사 = {
    '어요','해요','하고','게요','이고','이다','이에','있고','어있','고요',
    '이요','네요','에요','라고','으로','에서','지만','는데','으면','아요',
    '겠어','하여','하며','이며','이라','이나','라서','아서','어서','해서',
    '하면','이면','라면','에게','으며','으나','이야','이죠','했어','됐어',
    '갔어','왔어','있어','없어','같아','싶어','봐요','줘요','하죠','이죠',
    '라죠','하네','이네','라네','다네','했네','됐네',
}

def pattern(p):
    """
    의미 있는 n-gram 패턴인지 판단합니다.
    걸러내는 경우:
      - 공백 제거 후 1글자 이하 (예: ' 불', ' 다')
      - 순수 조사·어미만으로 구성 (예: '어요', '하고', '해요')
      - 끝 공백 포함 중복 패턴 (예: '어요 ' → '어요'와 동일)
    """
    p_clean = p.strip()
    if len(p_clean) <= 1:                # 1글자 이하
        return False
    if p_clean in 어미_조사:               # 순수 어미·조사
        return False
    if p.endswith(' ') and p.strip() in 어미_조사:   # 공백 포함 어미
        return False
    return True

print("의미 없는 패턴 필터 준비 완료!")
print()
print("예시:")
text = ['어요', '어요 ', ' 맛있', '바삭', '하고', '치킨', '식었', ' 불', '불친절']
for p in text:
    result = "✅ 유지" if pattern(p) else "❌ 제거"
    print(f"  {result}: '{p}'")

의미 없는 패턴 필터 준비 완료!

예시:
  ❌ 제거: '어요'
  ❌ 제거: '어요 '
  ✅ 유지: ' 맛있'
  ✅ 유지: '바삭'
  ❌ 제거: '하고'
  ✅ 유지: '치킨'
  ✅ 유지: '식었'
  ❌ 제거: ' 불'
  ✅ 유지: '불친절'


In [33]:
feature_names = trans.get_feature_names_out()
weight = LR.coef_[0]

# 의미 있는 패턴만 필터링
index = [i for i, p in enumerate(feature_names) if pattern(p)]
filtered_feature  = feature_names[index]
filtered_weight   = weight[index]

# 긍정에 강한 패턴 TOP 10
p_index = filtered_weight.argsort()[-10:][::-1]

print("긍정에 강한 패턴 TOP 10  (+ 클수록 긍정에 영향):")
print("-" * 42)
for i in p_index:
    bar = "█" * min(int(filtered_weight[i] * 3), 20)
    print(f"  '{filtered_feature[i]}':  {filtered_weight[i]:+.3f}  {bar}")

긍정에 강한 패턴 TOP 10  (+ 클수록 긍정에 영향):
------------------------------------------
  '맛있':  +0.234  
  ' 맛있':  +0.234  
  '있어요 ':  +0.205  
  '있어요':  +0.205  
  ' 맛있어':  +0.167  
  '맛있어요':  +0.167  
  '맛있어':  +0.167  
  '신선':  +0.156  
  ' 신선':  +0.156  
  '찌개 ':  +0.143  


In [34]:
# 부정에 강한 패턴 TOP 10
n_index = filtered_weight.argsort()[:10]

print("부정에 강한 패턴 TOP 10  (- 클수록 부정에 영향):")
print("-" * 42)
for i in n_index:
    막대 = "█" * min(int(abs(filtered_weight[i]) * 3), 20)
    print(f"  '{filtered_feature[i]}':  {filtered_weight[i]:+.3f}  {막대}")

부정에 강한 패턴 TOP 10  (- 클수록 부정에 영향):
------------------------------------------
  ' 너무 ':  -0.220  
  '너무 ':  -0.220  
  ' 너무':  -0.220  
  '너무':  -0.220  
  '었어요 ':  -0.156  
  '었어':  -0.156  
  '었어요':  -0.156  
  '눅눅':  -0.127  
  ' 눅눅':  -0.127  
  '망이':  -0.123  


---
## 도전 과제 — 영화 리뷰 감성 분석

**목표:** STEP 0의 핵심 코드를 영화 리뷰 데이터에 그대로 적용합니다.  
코드 구조는 완전히 동일합니다. **데이터만 바꾸면 됩니다.**

>  **배달 vs 영화 비교**  
> 같은 파이프라인이지만 다른 가중치를 학습합니다.  
> 배달: "바삭", "빠르다" → 긍정 / "식다", "늦다" → 부정  
> 영화: "감동", "연기", "훌륭" → 긍정 / "지루", "조잡", "허망" → 부정  
> 이것이 머신러닝의 핵심 — **데이터가 바뀌면 모델이 스스로 새 패턴을 학습합니다.**

In [36]:
# 영화 리뷰 데이터
p_reviews = [
    "연기가 정말 훌륭하고 스토리도 감동적이에요",
    "배우들의 연기력이 뛰어나고 영상미도 아름다워요",
    "스토리 전개가 탄탄하고 반전도 훌륭했어요",
    "음악이 영화 분위기와 완벽하게 어울려요",
    "감독의 연출이 세밀하고 캐릭터도 잘 살렸어요",
    "두 시간이 전혀 지루하지 않았어요 최고의 영화",
    "몰입감이 대단하고 결말이 인상적이었어요",
    "배우 케미가 좋고 대사도 자연스러워요",
    "영상미가 뛰어나고 OST도 감동적이에요",
    "현실적인 스토리와 훌륭한 연기가 인상적이에요",
]

n_reviews = [
    "시나리오가 진부하고 연기도 어색했어요",
    "스토리가 너무 예측 가능하고 지루했어요",
    "특수효과만 화려하고 내용이 없어요",
    "결말이 너무 허망하고 개연성이 부족해요",
    "두 시간이 너무 길게 느껴지고 집중이 안 됐어요",
    "배우 캐스팅이 잘못됐고 대사도 어색했어요",
    "스토리가 산으로 가고 결말도 이해가 안 돼요",
    "기대했는데 실망이에요 돈이 아까워요",
    "연출이 조잡하고 CG도 조악해요",
    "주인공 캐릭터가 매력 없고 공감이 안 돼요",
]

X_m = p_reviews + n_reviews
y_m = [1]*10 + [0]*10

print(f"영화 리뷰: {len(X_m)}개 (긍정 {sum(y_m)}개, 부정 {len(y_m)-sum(y_m)}개)")

영화 리뷰: 20개 (긍정 10개, 부정 10개)


In [45]:
# STEP 1: 전처리
X_m = [preproc(r) for r in X_m]

# STEP 2: 데이터 분할
X_tr, X_te, y_tr, y_te = train_test_split(
    X_m, y_m, test_size=0.25, random_state=42)

print(f"train set: {len(X_tr)}개 / test set: {len(X_te)}개")

# STEP 3: TF-IDF 변환
trans2 = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4))
X_tr_vec = trans2.fit_transform(X_tr)
X_te_vec = trans2.transform(X_te)

print(f"feature 수: {X_tr_vec.shape[1]}개")

# STEP 4: 두 모델 학습 및 예측
model1 = MultinomialNB()
model1.fit(X_tr_vec, y_tr)
pred1 = model1.predict(X_te_vec)

model2 = LogisticRegression(max_iter=1000, random_state=42)
model2.fit(X_tr_vec, y_tr)
pred2 = model2.predict(X_te_vec)

# STEP 5: 성능 비교
print("="*25)
print(f"나이브 베이즈:  {accuracy_score(y_te, pred1):.2%}" )
print(f"로지스틱 회귀: {accuracy_score(y_te, pred2):.2%}" )
print("="*25)

# STEP 6: 상세 성능
print("나이브 베이즈 상세:")
print(classification_report(y_te, pred1,
                              target_names=["부정","긍정"], zero_division=0))
print("="*60)
print("로지스틱 회귀 상세:")
print(classification_report(y_te, pred2,
                              target_names=["부정","긍정"], zero_division=0))
print("="*60)

train set: 15개 / test set: 5개
feature 수: 580개
나이브 베이즈:  60.00%
로지스틱 회귀: 80.00%
나이브 베이즈 상세:
              precision    recall  f1-score   support

          부정       0.00      0.00      0.00         2
          긍정       0.60      1.00      0.75         3

    accuracy                           0.60         5
   macro avg       0.30      0.50      0.38         5
weighted avg       0.36      0.60      0.45         5

로지스틱 회귀 상세:
              precision    recall  f1-score   support

          부정       1.00      0.50      0.67         2
          긍정       0.75      1.00      0.86         3

    accuracy                           0.80         5
   macro avg       0.88      0.75      0.76         5
weighted avg       0.85      0.80      0.78         5



In [49]:
# STEP 7: 영화 리뷰 긍정/부정 핵심 패턴
feat2 = trans2.get_feature_names_out()
coef2 = model2.coef_[0]

idx2 = [i for i, p in enumerate(feat2) if pattern(p)]
filtered_feat2   = feat2[idx2]
filtered_coef2   = coef2[idx2]

print("="*30)
print("영화 리뷰 — 긍정에 강한 패턴 TOP 8:")
for i in filtered_coef2.argsort()[-8:][::-1]:
    print(f"  '{filtered_feat2[i]}': {filtered_coef2[i]:+.3f}")
print()
print("="*30)
print("영화 리뷰 — 부정에 강한 패턴 TOP 8:")
for i in filtered_coef2.argsort()[:8]:
    print(f"  '{filtered_feat2[i]}': {filtered_coef2[i]:+.3f}")

영화 리뷰 — 긍정에 강한 패턴 TOP 8:
  ' 훌륭': +0.108
  '훌륭': +0.108
  ' 인상적': +0.108
  '인상적이': +0.108
  '인상': +0.108
  ' 인상': +0.108
  '적이': +0.108
  '인상적': +0.108

영화 리뷰 — 부정에 강한 패턴 TOP 8:
  ' 너무': -0.135
  ' 너무 ': -0.135
  '너무': -0.135
  '너무 ': -0.135
  '리가 ': -0.102
  '리가': -0.102
  '스토리가': -0.102
  '토리가 ': -0.102
